In [10]:
!pip install numpy qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 85.8 MB/s eta 0:00:00:00:0100:01


In [4]:
!pip install git+https://github.com/stephenhky/QAlgo.git@develop

  Cloning https://github.com/stephenhky/QAlgo.git (to revision develop) to /tmp/pip-req-build-jgol8p1h
  Running command git clone --filter=blob:none --quiet https://github.com/stephenhky/QAlgo.git /tmp/pip-req-build-jgol8p1h
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  Branch 'develop' set up to track remote branch 'develop' from 'origin'.
  Resolved https://github.com/stephenhky/QAlgo.git to commit bd5cb4df09c6cbef25d05315b6ddb28871a61538
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for qalgo: filename=qalgo-0.0.2a1-py3-none-any.whl size=4696 sha256=b0b80c1a38ab2968adb8f2b7f1f8d8b7bf952ecfa4fa9421ae57c1da0a22ce2f
  Stored in directory: /tmp/pip-ephem-wheel-cache-t7g4di4i/wheels/18/07/e8/d052d744e1e52c4cbba496a94efc1f79bda87751160b756285
Successfully built qalgo


In [16]:
from typing import Annotated

import numpy as np
import numpy.typing as npt
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister, Gate
from qiskit import transpile
from qiskit.circuit.library import StatePreparation, HamiltonianGate
from qiskit.quantum_info import Statevector
from qiskit_aer import StatevectorSimulator

from qalgo.phase import PhaseEstimationGate, InversePhaseEstimationGate

In [ ]:
sv_check = [None, None, None, None, None, None]

def HHLGate(
        A: Annotated[npt.NDArray[np.float64], "2D Hermitian matrix"],
        b: Annotated[npt.NDArray[np.float64], "1D array"],
        nb_b: int,   # b qubit
        nb_clock: int,   # clock qubit
        nb_ancilla: int = 1,    # ancilla qubit
        time: float | np.float64 = 2*np.pi/4
) -> tuple[Gate, Statevector]:
    assert nb_ancilla == 1     # ancilla must have only one qubit
    np.testing.assert_array_almost_equal(A, A.conj().T)   # test Hermitianity

    # initiating the circuit
    b_register = QuantumRegister(nb_b)
    clock_register = QuantumRegister(nb_clock)
    ancilla_register = QuantumRegister(nb_ancilla)
    qc = QuantumCircuit(ancilla_register, clock_register, b_register)
    sv_check[0] = Statevector(qc)

    # initiating b
    b_state_prep = StatePreparation(b)
    qc.append(b_state_prep, [b_register[i] for i in range(nb_b)])
    sv_check[1] = Statevector(qc)

    # initiating evolution gate
    evolution_gate = HamiltonianGate(A, time=time)
    qc.append(evolution_gate.control(1), [clock_register[i] for i in range(nb_clock)])
    sv_check[2] = Statevector(qc)

    # forward quantum phase estimation
    qc.append(
        PhaseEstimationGate(evolution_gate, nb_clock, nb_b),
        [clock_register[i] for i in range(nb_clock)] + [b_register[i] for i in range(nb_b)]
    )
    sv_check[3] = Statevector(qc)

    # controlled rotation
    for i in range(nb_clock):
        qc.cry(np.pi / (2**i), [clock_register[i] for i in range(nb_clock)], ancilla_register[0])
    sv_check[4] = Statevector(qc)

    # reverse quantum phase estimation
    qc.append(
        InversePhaseEstimationGate(evolution_gate.inverse(), nb_clock, nb_b),
        [clock_register[i] for i in range(nb_clock)] + [b_register[i] for i in range(nb_b)]
    )
    sv_check[5] = Statevector(qc)

    return qc.to_gate()

In [37]:
A = np.array([[1., -np.reciprocal(3.)], [-np.reciprocal(3.), 1.]])
b = np.array([0., 1.])

nb_b = 1
nb_clock = 4

ancilla_register = QuantumRegister(1)
clock_register = QuantumRegister(nb_clock)
b_register = QuantumRegister(nb_b)
# classical_register = ClassicalRegister(1)
qc = QuantumCircuit(ancilla_register, clock_register, b_register)
sv0 = Statevector(qc)
qc.append(
    HHLGate(A, b, nb_b, nb_clock),
    [ancilla_register[0]] + [clock_register[i] for i in range(nb_clock)] + [b_register[i] for i in range(nb_b)]
)

sv1 = Statevector(qc)

CircuitError: 'The amount of qubit(4)/clbit(0) arguments does not match the gate expectation (1).'

In [35]:
sv_check[2].draw("latex")

<IPython.core.display.Latex object>

In [27]:
sv0.draw("latex")

<IPython.core.display.Latex object>

In [18]:
sv1.draw("latex")

<IPython.core.display.Latex object>

In [23]:
filtered_probs = {
    str(key): float(prob)
    for key, prob in sv1.probabilities_dict().items()
    if key[-1] == '1'
}

In [24]:
{
    "0": sum(val for key, val in filtered_probs.items() if key[0]=="0"),
    "1": sum(val for key, val in filtered_probs.items() if key[0]=="1")
}

{'0': 0.1296869286407786, '1': 0.14832809982527415}

In [25]:
filtered_probs

{'000001': 9.323695588926283e-07,
 '000011': 0.002304847970437607,
 '000101': 0.011085581418922621,
 '000111': 0.002400892387697588,
 '001001': 0.012558009667678775,
 '001011': 0.005426116690718892,
 '001101': 0.018703002302701322,
 '001111': 0.032604116349693506,
 '010001': 0.0038560775138132006,
 '010011': 0.00047910016553311336,
 '010101': 0.00021540661412698003,
 '010111': 0.0014610409671066676,
 '011001': 0.0025775717044666064,
 '011011': 0.0038250353714755925,
 '011101': 0.028014661842689154,
 '011111': 0.004174535304158098,
 '100001': 0.005820676644661509,
 '100011': 0.000852732737652018,
 '100101': 0.0001370415497944893,
 '100111': 0.004480522907962066,
 '101001': 0.003592268673102607,
 '101011': 0.014020437632095933,
 '101101': 0.07815745295948158,
 '101111': 0.007345416330508098,
 '110001': 0.00021218754110327267,
 '110011': 0.0014274688426777033,
 '110101': 0.005031843164310783,
 '110111': 0.0008684068873420695,
 '111001': 0.007097917566425714,
 '111011': 0.00116892644508712